# Real vs guessed AQI: forest B on one chart

**The idea:** the poster's model, **forest B**, guesses **tomorrow's** AQI in a county from today's AQI, weather and date. This notebook draws those guesses next to the **real** AQI, day by day, for one Ohio county. It is the kind of chart used for forecasts: the days the model learned from are **dashed**, the days it has never seen are **solid**, and a vertical line shows where one ends and the other starts.

**How to run:** choose **Runtime > Run all**. When the first cell asks, upload the Excel file `AQI_Ohio_2022_2026_with_weather.xlsx`. It takes about 1 minute.

**What is inside**
- **Steps 1-3: get the data ready.** Load the file, make the answer and the clues, split the days into train days and test days. These are the same 20 clues and the same split as the poster.
- **Step 4: train forest B** and let it guess every day.
- **Steps 5-6: the chart.** First the table behind it, then the Altair chart for the whole file.
- **Step 7: zoom in** on the last 120 days.
- **Step 8: the numbers** behind the chart: the error on the train days, on the test days, and the Difference.

**Words you will see**
- **Real AQI**: the AQI that was measured on that day (the blue line).
- **Guess**: what forest B said the day before (the orange line).
- **Train days / test days**: the model learns from the train days (2022-2024). We grade it on the test days (2025 to May 2026), which it has never seen.
- **MAE**: how many AQI points the guess is wrong by, on average. Smaller is better.
- **RMSE**: like MAE, but big mistakes count extra. Smaller is better.
- **Difference**: test error minus train error. A big positive Difference is the warning sign of overfitting.
- **Altair**: the Python library that draws the chart. Colab normally has it already.

**Note:** the numbers in the text come from our run. If your Colab has other library versions, a number can differ in the second decimal.

## Step 1. Get the data

Run the next cell. Click **Choose files** and pick the Excel file. Each row is one county on one day: its AQI, the main pollutant and the weather.

In [ ]:
import numpy as np
import pandas as pd
import altair as alt
from google.colab import files

alt.data_transformers.disable_max_rows()              # Altair stops at 5,000 rows by default. This lifts the limit.

uploaded = files.upload()                             # a "Choose files" button appears: pick the Excel file
filename = list(uploaded)[0]                          # the name of the file you uploaded
df = pd.read_excel(filename)                          # read the Excel file into a table called df
df = df.rename(columns={"county Name": "County"})     # a shorter column name
df = df.sort_values(["County", "Date"]).reset_index(drop=True)   # one county after another, oldest day first

## Step 2. The answer and the clues

For every day we want to guess **tomorrow's AQI**. That is the **answer**. The **clues** are facts we know on that day:
- today's AQI, the AQI 1, 2 and 6 days ago, and the averages of the last 3 and the last 7 days;
- the time of year, the weekday of tomorrow, whether ozone set today's AQI, and the wind direction;
- 7 weather columns and the county's usual AQI (the last clue is added in Step 3).

Every calculation stays inside one county, so "yesterday" never comes from another county.

In [ ]:
weather = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)",
           "Wind Gust Max (mph)", "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]       # the 7 weather columns

each_county = df.groupby("County")                    # every calculation below stays inside one county

df["tomorrow"] = each_county.AQI.shift(-1)            # the ANSWER: the AQI on the next row
df["aqi_today"] = df.AQI
df["aqi_yesterday"] = each_county.AQI.shift(1)        # the AQI on the row before
df["aqi_2_days_ago"] = each_county.AQI.shift(2)
df["aqi_6_days_ago"] = each_county.AQI.shift(6)
df["aqi_3_day_average"] = each_county.AQI.transform(lambda day: day.rolling(3).mean())   # today and the 2 days before
df["aqi_7_day_average"] = each_county.AQI.transform(lambda day: day.rolling(7).mean())   # today and the 6 days before

In [ ]:
angle = 2 * np.pi * df.Date.dt.dayofyear / 365.25                  # the day of the year as an angle on a circle
df["day_of_year_sin"] = np.sin(angle)
df["day_of_year_cos"] = np.cos(angle)
df["tomorrow_weekday"] = (df.Date + pd.Timedelta(days=1)).dt.dayofweek          # 0 = Monday ... 6 = Sunday
df["ozone_main"] = (df["Defining Parameter"] == "Ozone").astype(int)            # 1 if ozone set today's AQI

wind_angle = np.deg2rad(df["Wind Direction (°)"])                  # the wind direction is also an angle on a circle
df["wind_dir_sin"] = np.sin(wind_angle)
df["wind_dir_cos"] = np.cos(wind_angle)

In [ ]:
early_clues = ["aqi_today", "aqi_yesterday", "aqi_2_days_ago", "aqi_6_days_ago", "aqi_3_day_average", "aqi_7_day_average",
               "day_of_year_sin", "day_of_year_cos", "tomorrow_weekday", "ozone_main", "wind_dir_sin", "wind_dir_cos"]
clues_without_county = early_clues + weather       # 19 clues. The 20th, county_average, needs the split of Step 3.

# Keep only the days that have every clue, and where the next row really is the next day
next_row_is_tomorrow = (each_county.Date.shift(-1) - df.Date).dt.days == 1
df = df[next_row_is_tomorrow].dropna(subset=clues_without_county + ["tomorrow"])
print("Days we can use:", len(df))

## Step 3. Train days and test days

The model learns from the **train days** (before 2025). We check it on the **test days** (2025 to May 2026), which it never sees while learning. The 20th clue, the county's usual AQI, is worked out from the train days only.

In [ ]:
train = df[df.Date < "2025-01-01"].copy()        # 2022-2024: the computer learns from these days
test = df[df.Date >= "2025-01-01"].copy()        # 2025 to May 2026: we check its guesses on these days

usual = train.groupby("County").tomorrow.mean()           # each county's usual AQI, from the training days only
train["county_average"] = train.County.map(usual)
test["county_average"] = test.County.map(usual)           # the test days get the training days' numbers: no peeking

clues = early_clues + ["county_average"] + weather        # the 20 clues, in the same order as on the poster
X_train, y_train = train[clues], train.tomorrow           # X = the clues, y = the answers
X_test, y_test = test[clues], test.tomorrow
print("Training days:", len(train), "| Test days:", len(test))

## Step 4. Train forest B and let it guess every day

Forest B is the random forest of the poster: 300 trees, at most 12 questions deep, at least 10 days in every leaf, and each question looks at half of the clues. It learns from the train days only. Then it makes a guess for **every** day, train days and test days.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

forest_b = RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=10, max_features=0.5, random_state=42, n_jobs=-1)
forest_b.fit(X_train, y_train)                        # learn from the train days only

train["guess"] = forest_b.predict(X_train)            # a guess for every train day
test["guess"] = forest_b.predict(X_test)              # a guess for every test day (days it has never seen)

print("Test days: MAE", round(mean_absolute_error(y_test, test.guess), 2), "| RMSE", round(mean_squared_error(y_test, test.guess) ** 0.5, 2))

**What this means:** on the test days forest B is wrong by 8.56 AQI points on average (MAE), and its RMSE is 12.03. These are the numbers on the poster. Now we draw its guesses.

## Step 5. The table behind the chart

A chart with two lines needs a **long table**: one row per day and per line. Its columns:
- `Day`: the day the guess is **about**. A guess is made on one day for the next day, so we move every row one day forward.
- `split`: `Train` or `Test`.
- `key`: the line the row belongs to, `Real AQI` or `Forest B's guess`.
- `AQI`: the value.

We draw one county. Stark County has data for every day of the file. To see another county, change the name in the cell and run Steps 5 to 8 again. Cuyahoga, Hamilton, Franklin and Medina also have more than 1,500 days.

In [ ]:
train["split"] = "Train"                                          # a label for every day: Train or Test
test["split"] = "Test"
both = pd.concat([train, test])                                   # all days in one table
both["Day"] = both.Date + pd.Timedelta(days=1)                    # a guess is made on a day, but it is about the NEXT day

county = "Stark"                                                  # change this name to see another county
one = both[both.County == county]
print(county, "has", len(one), "days, from", one.Day.min().date(), "to", one.Day.max().date())

long = one[["Day", "split", "tomorrow", "guess"]].melt(id_vars=["Day", "split"], var_name="key", value_name="AQI")      # two columns become two lines
long["key"] = long["key"].map({"tomorrow": "Real AQI", "guess": "Forest B's guess"})      # nicer names for the legend
long.sample(4, random_state=0)                                    # four random rows of the long table

## Step 6. The chart

The chart is built from two Altair charts, one on top of the other:
1. `lines`: the two lines. `color` gives one line to each `key` (blue = real, orange = guess). `strokeDash` makes the `split` visible: `[5, 3]` is a dashed line (5 pixels of line, 3 of gap), `[1, 0]` has no gap, so it is solid.
2. `split_line`: a vertical dashed line on the first test day.

The `+` puts `split_line` on top of `lines`.

To keep the chart as a picture, click the three dots at its top right and choose **Save as PNG**.

In [ ]:
line_colors = alt.Scale(domain=["Real AQI", "Forest B's guess"], range=["#2a78d6", "#eb6834"])   # blue = real, orange = guess
line_dashes = alt.Scale(domain=["Train", "Test"], range=[[5, 3], [1, 0]])                       # train: dashed, test: solid

lines = alt.Chart(long).mark_line(strokeWidth=1).encode(
    x=alt.X("Day:T", title="Day being guessed"),
    y=alt.Y("AQI:Q", title="AQI"),
    color=alt.Color("key:N", scale=line_colors, title=None),                  # one line for the real AQI, one for the guess
    strokeDash=alt.StrokeDash("split:N", scale=line_dashes, title=None),      # dashed on the train days, solid on the test days
)
split_day = one[one.split == "Test"].Day.min()                                # the first test day
split_line = alt.Chart(pd.DataFrame({"Day": [split_day]})).mark_rule(color="black", strokeDash=[5, 4]).encode(
    x=alt.X("Day:T", title="Day being guessed"))

(lines + split_line).properties(title=f"Next-day AQI in {county} County: real vs forest B's guess", width=800, height=300)

**What this means** (for Stark County)
- The dashed part is the train days and the solid part is the test days. Forest B learned only from the dashed part. The black dashed line is where the test days start.
- The orange guess follows the blue real line on calm days, but it is flatter: it stays inside a narrower range. On the solid part the highest real AQI is 124 and the highest guess is 98.
- The orange line seems to come a little late, as if the blue line were moved one day to the right. Step 7 zooms in to show this.

## Step 7. Zoom in on the last 120 days

Each dot is one day. Look at a sharp up-and-down of the blue line: the orange line makes the same move **one day later**. The most useful clue is today's AQI, so the guess for tomorrow mostly follows today's AQI.

In [ ]:
last_120 = long[long.Day > long.Day.max() - pd.Timedelta(days=120)]        # the last 120 days of the file

alt.Chart(last_120).mark_line(point=True, strokeWidth=1).encode(
    x=alt.X("Day:T", title="Day being guessed"),
    y=alt.Y("AQI:Q", title="AQI"),
    color=alt.Color("key:N", scale=line_colors, title=None),
).properties(title=f"The last 120 days in {county} County", width=800, height=300)

Now a number for the same idea. On all the test days (all counties) we measure how far apart some pairs are, on average:

In [ ]:
print("Test days, all counties. Average distance, in AQI points, between:")
print("  the guess and TODAY's AQI:        ", round((test.guess - test.aqi_today).abs().mean(), 2))
print("  the guess and the REAL answer:    ", round((test.guess - test.tomorrow).abs().mean(), 2))
print("  TODAY's AQI and the REAL answer:  ", round((test.aqi_today - test.tomorrow).abs().mean(), 2), " <- the lazy guess: tomorrow = today")

**What this means**
- The guess is closer to today's AQI (6.25 points away) than to the real answer (8.56 points away). Forest B mostly follows today's AQI, adjusted with the recent averages, the weather and the date.
- That is still better than repeating today's AQI as it is: the lazy guess is 10.27 points away from the real answer.
- This is why the orange line misses sudden spikes: today's AQI, the weather and the date give too little warning of a spike that starts tomorrow.

## Step 8. The numbers behind the chart

The same comparison as a table. **Difference** = test error minus train error: a big positive Difference means overfitting. The table also holds the lazy baseline "tomorrow = today", which learns nothing.

In [ ]:
rows = []
for label, part, column in [("Baseline: tomorrow = today (all counties)", both, "aqi_today"),
                            ("Forest B (all counties)", both, "guess"),
                            (f"Forest B ({county} only)", one, "guess")]:
    tr, te = part[part.split == "Train"], part[part.split == "Test"]           # the train days and the test days of this group
    rows.append({"Model": label,
                 "Train MAE": mean_absolute_error(tr.tomorrow, tr[column]), "Test MAE": mean_absolute_error(te.tomorrow, te[column]),
                 "Train RMSE": mean_squared_error(tr.tomorrow, tr[column]) ** 0.5, "Test RMSE": mean_squared_error(te.tomorrow, te[column]) ** 0.5})
table = pd.DataFrame(rows).set_index("Model")
table["MAE Difference"] = table["Test MAE"] - table["Train MAE"]               # test minus train
table["RMSE Difference"] = table["Test RMSE"] - table["Train RMSE"]
table.round(2)

**What this means**
- Forest B beats the lazy baseline on the test days: MAE 8.56 against 10.27, and RMSE 12.03 against 14.57.
- Its Difference is small and positive (MAE +1.96, RMSE +2.68): it is a little worse on new days than on the days it studied. This is a mild gap, not a sign that it memorized the train days.
- The baseline's Difference is below 0. It learns nothing, so it cannot overfit, and its test days are a little calmer than its train days.
- Stark alone looks like the whole state: its average miss is 7.09 on the train days and 9.02 on the test days.

# Summary

- **The chart:** the real AQI (blue) against forest B's guess (orange) for one county, with the train days dashed, the test days solid and a vertical line where the test days start.
- **The model:** on the test days forest B is off by 8.56 AQI points on average (RMSE 12.03). That beats "tomorrow = today" (10.27 and 14.57), and the gap between the train days and the test days (the Difference) is small.
- **The limit:** the guess mostly tracks today's AQI, so the orange line runs one day behind the blue line and misses the spikes. Across all counties, 109 test days had a real AQI above 100, and forest B did not guess above 100 on any of them.
- **Next step:** add next-day weather forecasts and wildfire smoke data to anticipate high-AQI spikes (the Future Work box of the poster).